# Astrometric concordance: before / after and SNR-stratified PINN

This notebook merges the earlier `astrometry_before_after.ipynb` and
`08_visualize_concordance_fits.ipynb` into a single end-to-end analysis of
the v8 latent position head outputs, and adds a new SNR-stratified PINN
study.

**Parts**

1. **Summary before / after.** Per-band bar chart, radial histograms,
   decomposed statistics. Same as the old before/after notebook.
2. **Spatial field maps.** 9 bands x 4 columns: raw anchors, PINN on raw,
   head-residual anchors, PINN on head residual. Same as `08`. Shows that
   the head absorbs the coherent smooth field.
3. **Anchor population.** Per-band SNR distributions, spatial density by
   SNR tercile, per-tile high-SNR anchor count. Identifies how many tiles
   are anchor-poor in a classical (bright-only) workflow.
4. **SNR-stratified PINN refits.** Fit the PINN on `bright`, `mid`, `faint`
   slices of the raw anchors and of the head-residual anchors. Compares
   field amplitude vs anchor count across slices.
5. **Field recovery comparison.** Does the faint-only fit reproduce the
   bright-only field? Also: field implied by the head's own predictions
   (`raw - head_resid`) vs direct field fits.

**Question we are answering.** In ECDFS we have enough bright anchors that
the classical bright-only PINN fit already pins down the ~5 mas smooth
concordance field. In sparser fields, bright anchors may be too few to
constrain a smooth field. The latent head is a per-object centroid
denoiser conditioned on 10-band features, so faint galaxies -- plentiful
even where bright stars are not -- should become effective anchors once
the head corrects them. This notebook tests that idea quantitatively.


In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import torch
from astropy.io import fits
from astropy.table import Table
from astropy.wcs import WCS
from IPython.display import display
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.ndimage import map_coordinates
from scipy.stats import binned_statistic_2d

plt.rcParams["figure.dpi"] = 120
plt.rcParams["image.origin"] = "lower"


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for cand in [start, *start.parents]:
        if (cand / "models").exists() and (cand / "io").exists():
            return cand
    return start


ROOT = find_repo_root()
MODELS = ROOT / "models"
if str(MODELS) not in sys.path:
    sys.path.insert(0, str(MODELS))

from astrometry2.pinn_field_solver import evaluate_pinn_mesh, fit_pinn_field
from astrometry2.fit_direct_pinn import sky_to_tangent_plane

CKPT = ROOT / "models" / "checkpoints" / "latent_position_v8_no_psf"
OUT_DIR = ROOT / "io"

ANCHORS_PATH = CKPT / "anchors.npz"
FITS_PINN_RAW = CKPT / "concordance_pinn_raw_fixed.fits"
FITS_PINN_HEAD = CKPT / "concordance_pinn_head_resid_fixed.fits"
FITS_NN_RAW = CKPT / "concordance_nn_raw_v2_fixed.fits"

for label, p in [
    ("Anchors", ANCHORS_PATH),
    ("PINN raw FITS", FITS_PINN_RAW),
    ("PINN head-resid FITS", FITS_PINN_HEAD),
    ("NN raw FITS", FITS_NN_RAW),
]:
    rel = p.relative_to(ROOT) if p.exists() else p
    print(f"{label:22s}: {rel}  {'OK' if p.exists() else 'MISSING'}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch device          : {DEVICE}")

# Band order and short / FITS-extension / anchors-key mapping.
RUBIN_BANDS = ["u", "g", "r", "i", "z", "y"]
NISP_BANDS = ["nisp_Y", "nisp_J", "nisp_H"]
ALL_BANDS = RUBIN_BANDS + NISP_BANDS

# Band index for the PINN chromatic embedding (matches dataset.BAND_TO_IDX).
BAND_TO_IDX = {
    "u": 0, "g": 1, "r": 2, "i": 3, "z": 4, "y": 5,
    "nisp_Y": 6, "nisp_J": 7, "nisp_H": 8,
}
N_BANDS = 9


def display_band(b: str) -> str:
    return b.replace("nisp_", "NISP ")


def fits_ext(b: str) -> str:
    return b.replace("nisp_", "NISP_") if b.startswith("nisp_") else b.upper()


def mag_mas(offset_arcsec: np.ndarray) -> np.ndarray:
    return np.hypot(offset_arcsec[:, 0], offset_arcsec[:, 1]) * 1000.0


def med_mas(offset_arcsec: np.ndarray) -> float:
    return float(np.nanmedian(mag_mas(offset_arcsec)))


def p68_mas(offset_arcsec: np.ndarray) -> float:
    return float(np.nanpercentile(mag_mas(offset_arcsec), 68.0))


def mad_sigma(x: np.ndarray) -> float:
    return 1.4826 * float(np.nanmedian(np.abs(x - np.nanmedian(x))))


def vector_stats(arr_arcsec: np.ndarray) -> dict:
    arr_mas = np.asarray(arr_arcsec) * 1000.0
    mag = np.hypot(arr_mas[:, 0], arr_mas[:, 1])
    sys_ = np.hypot(np.nanmedian(arr_mas[:, 0]), np.nanmedian(arr_mas[:, 1]))
    return {
        "sys": float(sys_),
        "med": float(np.nanmedian(mag)),
        "p68": float(np.nanpercentile(mag, 68.0)),
        "mad_ra": mad_sigma(arr_mas[:, 0]),
        "mad_de": mad_sigma(arr_mas[:, 1]),
    }


## Part 1 -- Summary before / after

Load the anchor cache and sample the already-fitted PINN fields at each
source. Then produce the bar chart, histograms and decomposed-statistics
table that summarize how the head and the smooth field interact.


In [ ]:
anchors = np.load(ANCHORS_PATH, allow_pickle=True)
print(f"anchors.npz keys: {len(anchors.files)}")
for k in sorted(anchors.files)[:6]:
    print(f"  {k}: shape={anchors[k].shape} dtype={anchors[k].dtype}")
print("  ...")


def sample_field_at_sources(fits_path: Path, band: str, ra, dec) -> np.ndarray | None:
    ext = fits_ext(band)
    with fits.open(fits_path) as hdul:
        try:
            dra_hdu = hdul[f"{ext}.DRA"]
            dde_hdu = hdul[f"{ext}.DDE"]
        except KeyError:
            return None
        dra_map = np.asarray(dra_hdu.data, dtype=np.float64)
        dde_map = np.asarray(dde_hdu.data, dtype=np.float64)
        w = WCS(dra_hdu.header, naxis=2)
    px, py = w.wcs_world2pix(np.asarray(ra), np.asarray(dec), 0)
    coords = np.stack([py, px], axis=0)
    dra = map_coordinates(dra_map, coords, order=1, mode="nearest")
    dde = map_coordinates(dde_map, coords, order=1, mode="nearest")
    return np.stack([dra, dde], axis=1).astype(np.float32)


data = {}
header_fmt = (
    f'{"Band":>8} {"N":>8}  {"Raw":>8} {"Raw+PINN":>9} {"Head":>8} '
    f'{"Head+PINN":>10}  {"|Fraw|":>8} {"|Fhead|":>8}'
)
print("\n" + header_fmt)
print("-" * 88)
for band in ALL_BANDS:
    if f"{band}_raw" not in anchors or f"{band}_head_resid" not in anchors:
        print(f"{display_band(band):>8}  missing in anchors, skipping")
        continue
    ra = anchors[f"{band}_ra"]
    dec = anchors[f"{band}_dec"]
    raw = anchors[f"{band}_raw"].astype(np.float32)
    head = anchors[f"{band}_head_resid"].astype(np.float32)
    snr = (anchors[f"{band}_snr"]
           if f"{band}_snr" in anchors.files else np.full(len(raw), np.nan, dtype=np.float32))
    tiles = (anchors[f"{band}_tiles"]
             if f"{band}_tiles" in anchors.files else np.full(len(raw), "", dtype=object))
    raw_field = sample_field_at_sources(FITS_PINN_RAW, band, ra, dec)
    head_field = sample_field_at_sources(FITS_PINN_HEAD, band, ra, dec)
    raw_pinn = (raw - raw_field) if raw_field is not None else None
    head_pinn = (head - head_field) if head_field is not None else None
    # Also compute what the head predicted per source (= raw - head_resid).
    head_pred = (raw - head).astype(np.float32)
    data[band] = {
        "ra": ra, "dec": dec,
        "raw": raw, "head": head, "head_pred": head_pred, "snr": snr, "tiles": tiles,
        "raw_field": raw_field, "head_field": head_field,
        "raw_pinn": raw_pinn, "head_pinn": head_pinn,
        "n": len(raw),
    }
    print(
        f'{display_band(band):>8} {len(raw):8d}  '
        f'{med_mas(raw):8.1f} '
        f'{med_mas(raw_pinn) if raw_pinn is not None else np.nan:9.1f} '
        f'{med_mas(head):8.1f} '
        f'{med_mas(head_pinn) if head_pinn is not None else np.nan:10.1f}  '
        f'{med_mas(raw_field) if raw_field is not None else np.nan:8.1f} '
        f'{med_mas(head_field) if head_field is not None else np.nan:8.1f}'
    )

bands_present = [b for b in ALL_BANDS if b in data]
print(f"\n{len(bands_present)} bands loaded")


In [ ]:
# Bar chart: Raw vs Raw+PINN vs Head vs Head+PINN.
x = np.arange(len(bands_present))
width = 0.19
series = [
    ("Raw WCS", "raw", "#c84b4b"),
    ("Raw + PINN field", "raw_pinn", "#d18b20"),
    ("Latent head", "head", "#3f7fbf"),
    ("Head + residual PINN", "head_pinn", "#3d9b63"),
]
values = {
    key: [med_mas(data[b][key]) if data[b][key] is not None else np.nan for b in bands_present]
    for _, key, _ in series
}

fig, ax = plt.subplots(figsize=(13, 5.2))
for i, (label, key, color) in enumerate(series):
    xpos = x + (i - 1.5) * width
    bars = ax.bar(xpos, values[key], width, label=label, color=color, alpha=0.86)
    for bar, val in zip(bars, values[key]):
        if not np.isfinite(val):
            continue
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1.0,
            f"{val:.0f}",
            ha="center", va="bottom", fontsize=7,
        )
ax.set_xticks(x)
ax.set_xticklabels([display_band(b) for b in bands_present], fontsize=9)
ax.set_ylabel("Median |offset| [mas]", fontsize=11)
ax.set_title("Astrometric offset vs Euclid VIS: before and after correction")
ax.legend(fontsize=8.5, ncol=2, loc="upper right")
ax.grid(axis="y", alpha=0.28)
ax.set_ylim(0, max(values["raw"]) * 1.22)
plt.tight_layout()
plt.savefig(OUT_DIR / "astrometry_before_after.png", dpi=220, bbox_inches="tight")
plt.savefig(OUT_DIR / "astrometry_bar_chart.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
# Per-band radial histograms.
fig, axes = plt.subplots(3, 3, figsize=(15, 10.5))
axes_flat = axes.ravel()
for idx, band in enumerate(bands_present):
    ax = axes_flat[idx]
    d = data[band]
    raw_mag = mag_mas(d["raw"])
    head_mag = mag_mas(d["head"])
    raw_pinn_mag = mag_mas(d["raw_pinn"]) if d["raw_pinn"] is not None else None
    head_pinn_mag = mag_mas(d["head_pinn"]) if d["head_pinn"] is not None else None
    finite_ref = raw_mag[np.isfinite(raw_mag)]
    xlim = float(np.nanpercentile(finite_ref, 98.5)) if finite_ref.size else 100.0
    xlim = max(50.0, min(240.0, xlim * 1.15))
    bins = np.linspace(0, xlim, 70)
    ax.hist(raw_mag, bins=bins, alpha=0.34, color="#c84b4b", density=True,
            label=f"Raw med={np.nanmedian(raw_mag):.1f}")
    if raw_pinn_mag is not None:
        ax.hist(raw_pinn_mag, bins=bins, alpha=0.34, color="#d18b20", density=True,
                label=f"Raw+PINN med={np.nanmedian(raw_pinn_mag):.1f}")
    ax.hist(head_mag, bins=bins, alpha=0.46, color="#3f7fbf", density=True,
            label=f"Head med={np.nanmedian(head_mag):.1f}")
    if head_pinn_mag is not None:
        ax.hist(head_pinn_mag, bins=bins, alpha=0.46, color="#3d9b63", density=True,
                label=f"Head+PINN med={np.nanmedian(head_pinn_mag):.1f}")
    ax.set_title(display_band(band), fontsize=10)
    ax.set_xlabel("|offset| [mas]", fontsize=8)
    ax.set_xlim(0, xlim)
    ax.legend(fontsize=6.8)
    ax.grid(axis="y", alpha=0.2)
for ax in axes_flat[len(bands_present):]:
    ax.axis("off")
plt.suptitle("Per-source astrometric residual distributions", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "astrometry_histograms.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:
# Decomposed statistics table: |sys|, med, p68, MADxy per stage.
columns = [("Raw", "raw"), ("Raw+PINN", "raw_pinn"), ("Head", "head"), ("Head+PINN", "head_pinn")]
print(f'{"Band":>8} {"N":>8}  ' + "  ".join(f"{name:^25s}" for name, _ in columns))
print(f'{"":>8} {"":>8}  ' + "  ".join(f'{"|sys|":>6} {"med":>6} {"p68":>6} {"MADxy":>6}' for _ in columns))
print("-" * 128)
for band in bands_present:
    pieces = []
    for _, key in columns:
        arr = data[band].get(key)
        if arr is None:
            pieces.append(f"{np.nan:6.1f} {np.nan:6.1f} {np.nan:6.1f} {np.nan:6.1f}")
            continue
        st = vector_stats(arr)
        madxy = 0.5 * (st["mad_ra"] + st["mad_de"])
        pieces.append(f'{st["sys"]:6.1f} {st["med"]:6.1f} {st["p68"]:6.1f} {madxy:6.1f}')
    print(f'{display_band(band):>8} {data[band]["n"]:8d}  ' + "  ".join(pieces))
print("\n|sys| = magnitude of median RA/Dec residual vector [mas]")
print("med/p68 = radial per-source residual magnitude [mas]")
print("MADxy = average robust 1D component scatter [mas]")


## Part 2 -- Spatial field maps

9 bands x 4 columns, taken from the old `08_visualize_concordance_fits`:

1. **raw anchors** -- binned median `|offset|` from `anchors.npz`.
2. **PINN raw** -- smooth PINN field on raw anchors (from the precomputed FITS).
3. **head-resid anchors** -- binned median of `_head_resid`; what is left per source after the head.
4. **PINN after head** -- smooth PINN field on head-residual anchors.

If the head absorbs the coherent component, the third column should look like scatter and the fourth column should be nearly flat.


In [ ]:
# 9x4 field comparison grid (from notebook 08). Uses the PINN raw FITS WCS to project anchors.
BIN_ARCSEC = 60.0
MIN_COUNT = 4
MAS = 1000.0

with fits.open(FITS_PINN_RAW) as hdul:
    img_hdr = next(h.header for h in hdul[1:] if h.data is not None and h.data.ndim == 2)
    img_shape = next(h.data for h in hdul[1:] if h.data is not None and h.data.ndim == 2).shape
wcs_out = WCS(img_hdr)
image_h_pix, image_w_pix = img_shape
bin_px = max(1, int(round(BIN_ARCSEC / float(img_hdr.get("DSTEP", 1.0)))))
x_edges = np.arange(0, image_w_pix + bin_px, bin_px)
y_edges = np.arange(0, image_h_pix + bin_px, bin_px)


def bin_anchors(band, field_arr):
    ra = data[band]["ra"]
    dec = data[band]["dec"]
    px, py = wcs_out.all_world2pix(ra, dec, 0)
    mag = np.hypot(field_arr[:, 0], field_arr[:, 1]) * MAS
    med, _, _, _ = binned_statistic_2d(px, py, mag, statistic="median", bins=[x_edges, y_edges])
    cnt, _, _, _ = binned_statistic_2d(px, py, mag, statistic="count", bins=[x_edges, y_edges])
    med = np.where(cnt >= MIN_COUNT, med, np.nan)
    return np.ma.masked_invalid(med.T), int(len(ra))


def load_fits_fields(fits_path):
    out = {}
    with fits.open(fits_path) as hdul:
        for hdu in hdul[1:]:
            if hdu.data is None or not hdu.name or "." not in hdu.name:
                continue
            prefix, field = hdu.name.upper().rsplit(".", 1)
            out.setdefault(prefix, {})[field] = np.asarray(hdu.data, dtype=float)
    return out


pinn_raw_maps = load_fits_fields(FITS_PINN_RAW)
pinn_head_maps = load_fits_fields(FITS_PINN_HEAD)

cmap = mpl.colormaps["magma"].copy()
cmap.set_bad("#202020")
col_labels = ["raw anchors", "PINN raw", "head-resid anchors", "PINN after head"]
vmax_fixed = 20.0  # mas

fig, axes = plt.subplots(len(bands_present), 4, figsize=(15, 2.2 * len(bands_present)), squeeze=False)
for row, band in enumerate(bands_present):
    ext = fits_ext(band)
    raw_bin, n_anch = bin_anchors(band, data[band]["raw"])
    head_bin, _ = bin_anchors(band, data[band]["head"])
    pinn_raw_mag = np.ma.masked_invalid(
        np.hypot(pinn_raw_maps[ext]["DRA"], pinn_raw_maps[ext]["DDE"]) * MAS
    )
    pinn_head_mag = np.ma.masked_invalid(
        np.hypot(pinn_head_maps[ext]["DRA"], pinn_head_maps[ext]["DDE"]) * MAS
    )
    panels = [
        (raw_bin, [0, image_w_pix, 0, image_h_pix]),
        (pinn_raw_mag, None),
        (head_bin, [0, image_w_pix, 0, image_h_pix]),
        (pinn_head_mag, None),
    ]
    for col, (arr, extent) in enumerate(panels):
        ax = axes[row, col]
        kwargs = dict(cmap=cmap, vmin=0, vmax=vmax_fixed, interpolation="nearest")
        if extent is not None:
            kwargs["extent"] = extent
        im = ax.imshow(arr, **kwargs)
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)
        if row == 0:
            ax.set_title(col_labels[col], fontsize=8, pad=4)
        v = arr.compressed() if np.ma.isMaskedArray(arr) else np.asarray(arr).ravel()
        v = v[np.isfinite(v)]
        med = np.nanmedian(v) if v.size else np.nan
        p95 = np.nanpercentile(v, 95) if v.size else np.nan
        extra = f"\n{n_anch} anch" if col == 0 else ""
        ax.text(
            0.03, 0.97, f"{ext}\nmed {med:.1f} | p95 {p95:.1f}{extra}",
            transform=ax.transAxes, va="top", ha="left", fontsize=6.5, color="white",
            bbox=dict(boxstyle="round,pad=0.22", facecolor="black", alpha=0.48, edgecolor="none"),
        )
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.04)
        cb = fig.colorbar(im, cax=cax)
        cb.ax.tick_params(labelsize=6)
fig.subplots_adjust(left=0.02, right=0.995, bottom=0.02, top=0.97, hspace=0.08, wspace=0.22)
fig.suptitle(f"raw anchors -> PINN raw | head-resid anchors -> PINN after head (bin={BIN_ARCSEC:.0f}\", vmax={vmax_fixed:.0f} mas)",
             fontsize=10, y=0.995)
plt.savefig(OUT_DIR / "concordance_3way.png", dpi=180, bbox_inches="tight")
plt.show()


## Part 3 -- Anchor population

A classical bright-only concordance workflow leans on high-SNR stars. How
many do we actually have per band and per tile? This section characterises
the anchor population so the SNR-stratified refits below are interpretable.

- Per-band SNR distributions (log-x).
- Spatial density by SNR tercile (which parts of the survey are
  anchor-rich for a given SNR cut).
- Per-tile counts of high-SNR anchors -- how many tiles would be
  anchor-poor in a classical workflow.


In [ ]:
# Per-band SNR distributions.
fig, axes = plt.subplots(3, 3, figsize=(14, 9))
for ax, band in zip(axes.ravel(), bands_present):
    snr = data[band]["snr"]
    snr = snr[np.isfinite(snr) & (snr > 0)]
    if snr.size == 0:
        ax.text(0.5, 0.5, "no SNR", transform=ax.transAxes, ha="center")
        ax.set_title(display_band(band))
        continue
    bins = np.logspace(np.log10(max(snr.min(), 1.0)), np.log10(max(snr.max(), 10.0)), 60)
    ax.hist(snr, bins=bins, color="#3f7fbf", alpha=0.85)
    ax.set_xscale("log")
    ax.set_xlabel("SNR", fontsize=8)
    med = np.median(snr)
    p10, p90 = np.percentile(snr, [10, 90])
    ax.axvline(med, color="k", lw=1, ls="--")
    ax.set_title(
        f"{display_band(band)}  N={snr.size}\n"
        f"SNR med={med:.0f}  p10={p10:.0f}  p90={p90:.0f}",
        fontsize=9,
    )
    ax.grid(axis="y", alpha=0.2)
for ax in axes.ravel()[len(bands_present):]:
    ax.axis("off")
plt.suptitle("Per-band SNR distribution", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "anchor_snr_distributions.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# Spatial density of anchors by SNR tercile (pooled across bands, i-band alone if present).

def snr_tercile_mask(snr_arr, band_snr_arr):
    """Return boolean masks (bright, mid, faint) using per-band terciles (top/mid/bottom third)."""
    good = np.isfinite(band_snr_arr) & (band_snr_arr > 0)
    if good.sum() < 30:
        return None
    lo, hi = np.percentile(band_snr_arr[good], [33.33, 66.67])
    bright = good & (band_snr_arr >= hi)
    mid = good & (band_snr_arr >= lo) & (band_snr_arr < hi)
    faint = good & (band_snr_arr < lo)
    return bright, mid, faint, lo, hi


# Show for i-band (high anchor count, well-measured). Same pattern applies to other bands.
band_for_map = "i" if "i" in data else bands_present[0]
d = data[band_for_map]
mask_tuple = snr_tercile_mask(d["snr"], d["snr"])
bright, mid, faint, lo_thr, hi_thr = mask_tuple

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)
ra_all = d["ra"]; dec_all = d["dec"]
extent = [ra_all.min(), ra_all.max(), dec_all.min(), dec_all.max()]
for ax, (name, m, color) in zip(
    axes,
    [
        (f"faint (SNR<{lo_thr:.0f})", faint, "#8b1a1a"),
        (f"mid ({lo_thr:.0f}-{hi_thr:.0f})", mid, "#d18b20"),
        (f"bright (SNR>={hi_thr:.0f})", bright, "#1d6ecf"),
    ],
):
    hb = ax.hexbin(ra_all[m], dec_all[m], gridsize=45, cmap="magma",
                   mincnt=1, extent=extent)
    ax.set_title(f"{display_band(band_for_map)}  {name}  N={m.sum()}", fontsize=10)
    ax.set_xlabel("RA [deg]")
    plt.colorbar(hb, ax=ax, label="anchors / bin")
    ax.invert_xaxis()
axes[0].set_ylabel("Dec [deg]")
plt.suptitle(f"Spatial density by SNR tercile ({display_band(band_for_map)})", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "anchor_density_by_snr.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# How many bright anchors per tile? A classical bright-only workflow struggles on low-count tiles.
SNR_BRIGHT_ABS = 30.0  # absolute SNR cut for the "bright classical anchor" definition

fig, axes = plt.subplots(3, 3, figsize=(14, 9))
for ax, band in zip(axes.ravel(), bands_present):
    snr = data[band]["snr"]
    tiles = data[band]["tiles"]
    if snr.size == 0 or tiles.size == 0:
        ax.axis("off"); continue
    good = np.isfinite(snr) & (snr > SNR_BRIGHT_ABS)
    tiles_g = tiles[good]
    if tiles_g.size == 0:
        ax.text(0.5, 0.5, "no bright anchors", transform=ax.transAxes, ha="center")
        ax.set_title(display_band(band))
        continue
    unique_t, counts = np.unique(tiles_g, return_counts=True)
    total_tiles = np.unique(tiles).size
    # zero pad tiles with no bright anchors
    counts_full = np.concatenate([counts, np.zeros(total_tiles - unique_t.size, dtype=int)])
    bins = np.arange(0, max(counts_full.max() + 2, 12))
    ax.hist(counts_full, bins=bins, color="#3d9b63", alpha=0.85)
    med = int(np.median(counts_full))
    zero = int((counts_full == 0).sum())
    poor = int((counts_full < 5).sum())
    ax.set_title(
        f"{display_band(band)}  tiles={total_tiles}\n"
        f"med={med}  zero={zero}  <5={poor}",
        fontsize=9,
    )
    ax.set_xlabel(f"bright (SNR>{SNR_BRIGHT_ABS:.0f}) anchors / tile", fontsize=8)
    ax.grid(axis="y", alpha=0.2)
for ax in axes.ravel()[len(bands_present):]:
    ax.axis("off")
plt.suptitle(f"Bright-anchor count per tile (SNR > {SNR_BRIGHT_ABS:.0f})", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "bright_anchors_per_tile.png", dpi=180, bbox_inches="tight")
plt.show()


## Part 4 -- SNR-stratified PINN refits

The main experiment. For each SNR slice (per-band tercile) we refit the
PINN jointly across all bands (same solver as the CLI) and record the
resulting field on a common mesh. Each fit is done twice:

- **on raw anchors** (`{band}_raw`): field amplitude reflects the smooth
  concordance signal plus centroid noise. Bright-only should be dominated
  by signal; faint-only should be noise-dominated.
- **on head-residual anchors** (`{band}_head_resid`): field amplitude
  reflects what the per-object head did NOT absorb. Should be small for
  all slices if the head is genuinely removing the coherent component.

PINN hyperparameters below are lighter than the CLI default (4000 steps
instead of 8000, smaller collocation pool) so the notebook finishes in a
few minutes on GPU.


In [ ]:
# Build per-slice anchor tables, then fit PINN jointly across bands.
N_STEPS_REFIT = 4000
N_COLLOC_REFIT = 8000

# Common tangent-plane frame: compute ra0/dec0 from all anchors so every refit shares axes.
ra_all = np.concatenate([data[b]["ra"] for b in bands_present])
dec_all = np.concatenate([data[b]["dec"] for b in bands_present])
pos_all, RA0, DEC0 = sky_to_tangent_plane(ra_all, dec_all)
POS_MIN = pos_all.min(axis=0).astype(np.float32)
POS_MAX = pos_all.max(axis=0).astype(np.float32)
FIELD_W = int(np.ceil(float(POS_MAX[0] - POS_MIN[0]))) + 2
FIELD_H = int(np.ceil(float(POS_MAX[1] - POS_MIN[1]))) + 2
print(f"Common tangent frame: RA0={RA0:.4f} DEC0={DEC0:.4f}")
print(f"Field size (arcsec): {FIELD_W} x {FIELD_H}")


def tangent_shifted(ra, dec):
    cosdec = np.cos(np.deg2rad(DEC0))
    x = (np.asarray(ra) - RA0) * cosdec * 3600.0
    y = (np.asarray(dec) - DEC0) * 3600.0
    pos = np.stack([x, y], axis=1).astype(np.float32)
    return (pos - POS_MIN[None, :]).astype(np.float32)


def build_slice(snr_mode: str, offset_key: str) -> dict:
    """snr_mode in {'all','bright','mid','faint'}; offset_key in {'raw','head_resid'}.

    Returns dict with pos, offsets, weights, band_idx, n_per_band.
    """
    pos_list, off_list, w_list, bi_list, nper = [], [], [], [], {}
    for band in bands_present:
        d = data[band]
        snr = d["snr"]
        off = d["raw"] if offset_key == "raw" else d["head"]
        good = np.isfinite(snr) & (snr > 0)
        if good.sum() < 30:
            continue
        if snr_mode == "all":
            sel = good
        else:
            lo, hi = np.percentile(snr[good], [33.33, 66.67])
            if snr_mode == "bright":
                sel = good & (snr >= hi)
            elif snr_mode == "mid":
                sel = good & (snr >= lo) & (snr < hi)
            elif snr_mode == "faint":
                sel = good & (snr < lo)
            else:
                raise ValueError(snr_mode)
        pos = tangent_shifted(d["ra"][sel], d["dec"][sel])
        pos_list.append(pos)
        off_list.append(off[sel])
        # Weight ~ SNR^2 (King-scaling). Floor so per-slice weights aren't pathological.
        snr_s = np.clip(snr[sel], 3.0, None)
        w_list.append((snr_s ** 2).astype(np.float32))
        bi_list.append(np.full(int(sel.sum()), BAND_TO_IDX[band], dtype=np.int64))
        nper[band] = int(sel.sum())
    if not pos_list:
        return None
    return {
        "pos": np.concatenate(pos_list, axis=0),
        "offsets": np.concatenate(off_list, axis=0).astype(np.float32),
        "weights": np.concatenate(w_list, axis=0),
        "band_idx": np.concatenate(bi_list, axis=0),
        "n_per_band": nper,
    }


def fit_slice(slice_tab, tag):
    print(f"\n=== Fit {tag}  (N={len(slice_tab['pos'])}, bands={len(slice_tab['n_per_band'])}) ===")
    t0 = time.time()
    model, meta = fit_pinn_field(
        pos_arcsec=slice_tab["pos"],
        offsets_arcsec=slice_tab["offsets"],
        weights=slice_tab["weights"],
        band_indices=slice_tab["band_idx"],
        n_bands=N_BANDS,
        hidden_dim=128,
        n_layers=5,
        n_steps=N_STEPS_REFIT,
        lr=1e-3,
        lambda_curl=1.0,
        lambda_lapl=0.1,
        lambda_band=0.1,
        n_collocation=N_COLLOC_REFIT,
        device=DEVICE,
        verbose=False,
    )
    print(f"  done in {time.time() - t0:.1f}s  best_loss={meta['best_loss']:.4g}")
    return model, meta


SLICES = ["all", "bright", "mid", "faint"]
KINDS = ["raw", "head_resid"]
fits_store: dict[tuple[str, str], dict] = {}
for kind in KINDS:
    for snr_mode in SLICES:
        tab = build_slice(snr_mode, kind)
        if tab is None:
            continue
        model, meta = fit_slice(tab, f"{snr_mode} / {kind}")
        fits_store[(snr_mode, kind)] = {
            "model": model, "meta": meta, "table": tab,
        }
print(f"\nCompleted {len(fits_store)} PINN refits")


In [ ]:
# Evaluate each fit on the common mesh (geometric field + per-band) and record amplitudes.
def eval_mesh(model, meta, band_idx_int=None):
    res = evaluate_pinn_mesh(
        model, meta, FIELD_H, FIELD_W, dstep=1,
        pos_arcsec_anchors=None, band_idx=band_idx_int,
    )
    return res


def field_rms_mas(res):
    dra = res["dra"] * 1000.0
    ddec = res["ddec"] * 1000.0
    return float(np.sqrt(np.nanmean(dra ** 2 + ddec ** 2)))


def field_p95_mas(res):
    mag = np.hypot(res["dra"], res["ddec"]) * 1000.0
    return float(np.nanpercentile(mag, 95))


summary_rows = []
for (snr_mode, kind), entry in fits_store.items():
    tab = entry["table"]
    # Residual at anchors.
    model = entry["model"]; meta = entry["meta"]
    pos_n = (tab["pos"] - meta["pos_min"]) / meta["pos_scale"] * 2.0 - 1.0
    with torch.no_grad():
        xy_t = torch.tensor(pos_n, dtype=torch.float32)
        bi_t = torch.tensor(tab["band_idx"], dtype=torch.long)
        pred = model(xy_t, band_idx=bi_t).numpy() * meta["off_scale"]
    resid = tab["offsets"] - pred
    # Geometric (achromatic) field on the mesh.
    geo = eval_mesh(model, meta, band_idx_int=None)
    summary_rows.append({
        "slice": snr_mode, "kind": kind,
        "N": int(tab["pos"].shape[0]),
        "raw_med_mas": float(med_mas(tab["offsets"])),
        "resid_med_mas": float(med_mas(resid)),
        "field_rms_mas": field_rms_mas(geo),
        "field_p95_mas": field_p95_mas(geo),
    })
    entry["geo"] = geo
    entry["resid"] = resid

summary_tab = Table(rows=[(r["slice"], r["kind"], r["N"],
                           f"{r['raw_med_mas']:.1f}",
                           f"{r['resid_med_mas']:.1f}",
                           f"{r['field_rms_mas']:.1f}",
                           f"{r['field_p95_mas']:.1f}")
                          for r in summary_rows],
                   names=["slice", "kind", "N", "raw_med",
                          "resid_med", "field_rms", "field_p95"])
display(summary_tab)


## Part 5 -- Field recovery comparison

The sparse-field question in pictures.

- **Geometric field maps** for bright-only vs faint-only on raw anchors,
  and faint-only on head-residual anchors. If the head-as-amplifier idea
  holds, the bright-raw map is the gold standard; the faint-raw map is
  noise-dominated; and the faint-head-resid map is nearly flat (head has
  already absorbed the smooth component even at the faint end).
- **Field amplitude vs anchor count** shows whether removing bright
  anchors actually destroys the classical concordance measurement, and
  how the head changes that.
- **Head-implied field.** `head_pred = raw - head_resid` is what the head
  predicts per source. Its binned spatial pattern is the field the head
  has effectively "learned". Compare it to the bright-only raw PINN fit.


In [ ]:
# Side-by-side geometric field maps.
PANELS = [
    ("bright / raw", ("bright", "raw")),
    ("faint / raw",  ("faint",  "raw")),
    ("all / raw",    ("all",    "raw")),
    ("bright / head_resid", ("bright", "head_resid")),
    ("faint / head_resid",  ("faint",  "head_resid")),
    ("all / head_resid",    ("all",    "head_resid")),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
vmax_map = 15.0
for ax, (label, key) in zip(axes.ravel(), PANELS):
    if key not in fits_store:
        ax.axis("off"); continue
    geo = fits_store[key]["geo"]
    mag = np.hypot(geo["dra"], geo["ddec"]) * 1000.0
    im = ax.imshow(mag, cmap="magma", vmin=0, vmax=vmax_map, origin="lower",
                   extent=[0, FIELD_W, 0, FIELD_H], aspect="equal")
    n = fits_store[key]["table"]["pos"].shape[0]
    rms = np.sqrt(np.nanmean((geo["dra"] * 1000) ** 2 + (geo["ddec"] * 1000) ** 2))
    ax.set_title(f"{label}   N={n}   field RMS={rms:.1f} mas", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.04)
    plt.colorbar(im, cax=cax, label="|field| [mas]")
plt.suptitle("Geometric field |f(x,y)| for SNR-stratified refits", y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / "snr_stratified_fields.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# Field amplitude vs anchor count summary plot.
fig, ax = plt.subplots(figsize=(8, 5))
kind_style = {"raw": ("o", "#c84b4b"), "head_resid": ("s", "#3f7fbf")}
for (snr_mode, kind), entry in fits_store.items():
    n = entry["table"]["pos"].shape[0]
    rms = np.sqrt(np.nanmean(
        (entry["geo"]["dra"] * 1000) ** 2 + (entry["geo"]["ddec"] * 1000) ** 2
    ))
    marker, color = kind_style[kind]
    ax.scatter(n, rms, marker=marker, s=85, color=color, alpha=0.85, edgecolor="k", linewidth=0.6,
               label=f"{kind}" if snr_mode == "bright" else None)
    ax.annotate(f"{snr_mode}", (n, rms), xytext=(6, 4), textcoords="offset points", fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("Anchors in fit", fontsize=10)
ax.set_ylabel("Geometric field RMS [mas]", fontsize=10)
ax.set_title("Field amplitude vs anchor count, by SNR slice", fontsize=11)
ax.grid(alpha=0.3)
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(OUT_DIR / "snr_field_amp_vs_N.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# Head-implied field: fit a PINN to head_pred = raw - head_resid for all anchors.
# This turns the per-object head predictions into a smooth field we can compare
# visually to the bright-only raw PINN (the classical gold standard).

pos_hp = []
off_hp = []
bi_hp = []
w_hp = []
for band in bands_present:
    d = data[band]
    snr = d["snr"]
    good = np.isfinite(snr) & (snr > 0)
    if good.sum() < 30:
        continue
    pos_hp.append(tangent_shifted(d["ra"][good], d["dec"][good]))
    off_hp.append(d["head_pred"][good])
    bi_hp.append(np.full(int(good.sum()), BAND_TO_IDX[band], dtype=np.int64))
    w_hp.append(np.clip(snr[good], 3.0, None).astype(np.float32) ** 2)
pos_hp = np.concatenate(pos_hp)
off_hp = np.concatenate(off_hp).astype(np.float32)
bi_hp = np.concatenate(bi_hp)
w_hp = np.concatenate(w_hp)
print(f"head_pred anchor pool: {len(pos_hp)}")
t0 = time.time()
model_hp, meta_hp = fit_pinn_field(
    pos_arcsec=pos_hp, offsets_arcsec=off_hp, weights=w_hp,
    band_indices=bi_hp, n_bands=N_BANDS,
    n_steps=N_STEPS_REFIT, n_collocation=N_COLLOC_REFIT,
    device=DEVICE, verbose=False,
)
print(f"head_pred PINN fit: {time.time() - t0:.1f}s  best_loss={meta_hp['best_loss']:.4g}")
geo_hp = evaluate_pinn_mesh(model_hp, meta_hp, FIELD_H, FIELD_W, dstep=1, band_idx=None)
rms_hp = float(np.sqrt(np.nanmean((geo_hp["dra"] * 1000) ** 2 + (geo_hp["ddec"] * 1000) ** 2)))
print(f"head-implied field RMS: {rms_hp:.1f} mas")

# Compare side by side: bright / raw (classical), all / raw, head-implied.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
panels_hp = [
    ("bright / raw (classical)", fits_store.get(("bright", "raw"), {}).get("geo")),
    ("all / raw",                fits_store.get(("all", "raw"),    {}).get("geo")),
    ("head-implied (all sources)", geo_hp),
]
for ax, (label, geo) in zip(axes, panels_hp):
    if geo is None:
        ax.axis("off"); continue
    mag = np.hypot(geo["dra"], geo["ddec"]) * 1000.0
    im = ax.imshow(mag, cmap="magma", vmin=0, vmax=15, origin="lower",
                   extent=[0, FIELD_W, 0, FIELD_H], aspect="equal")
    rms = float(np.sqrt(np.nanmean((geo["dra"] * 1000) ** 2 + (geo["ddec"] * 1000) ** 2)))
    ax.set_title(f"{label}  RMS={rms:.1f} mas", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.04)
    plt.colorbar(im, cax=cax)
plt.suptitle("Classical bright-only vs all-anchors vs head-implied field", y=1.03)
plt.tight_layout()
plt.savefig(OUT_DIR / "head_implied_vs_classical_field.png", dpi=180, bbox_inches="tight")
plt.show()


## Reading the result

Things to check against the printed tables / figures:

1. **Bright-only / raw PINN** should give a field RMS close to the CLI
   reference (~5 mas geometric amplitude in ECDFS). This is the
   classical concordance measurement. If it matches, we trust it as the
   gold-standard WCS field.
2. **Faint-only / raw PINN** is the no-head sparse-field scenario. If its
   field amplitude is much larger than bright/raw, that amplitude is
   dominated by per-source centroid noise; the smooth field is not
   actually this big. This is exactly the "can't measure concordance
   without bright anchors" failure mode.
3. **Faint-only / head_resid PINN** tests whether the head has genuinely
   absorbed the coherent component at the faint end. If this is small and
   comparable to bright/head_resid, then the head works equally well for
   faint sources -- i.e. the per-object correction really is source-level,
   not just "it works on bright sources where centroids are already good."
4. **Head-implied field vs bright/raw.** If they agree, the head's
   predictions per-source behave like a dense sampling of the smooth
   field. In sparse fields you can use the head instead of a bright-only
   PINN fit to get a concordance correction.

Expected ECDFS outcome: bright/raw field ~ 5 mas, faint/raw field
dominated by scatter (>10 mas), head-resid slices all ~1 mas, and the
head-implied field matches bright/raw within a few mas. The sparse-field
value of the head is therefore quantified by the faint/raw fit, where it
turns a noise-dominated measurement into one that matches the classical
bright-only fit.
